# Práctica 2: Modelo cinemático inverso de un manipulador y planteamiento de trayectoria


## Objetivo



El objetivo de esta práctica es que el alumno comprenda, interprete y modifique el modelo de cinemática directa e inversa d eun moanipulador serial.


### Metas 

- Que el alumno aplique un modelo cinemático, obteniendo las matrices de transformación y la postura del manipulador 
- Que el alumno aplique un modelo cinemático inverso para calcular una trayectoria a partir de una posición actual hacia una posición final
- Que el alumno grafique y analice los resultados del modelo

### Contribución al perfil del egresado

La siguiente práctica contribuye en los siguientes puntos al perfil del egresado:

#### Aptitudes y habilidades

- Para modelar, simular e interpretar el comportamiento de los sistemas mecatrónicos.
- Para diseñar, construir, operar y mantener los sistemas mecatrónicos y sus componentes.

#### Actitudes

- Tener confianza en su preparación académica.
- Comprometido con su actualización, superación y competencia profesional.

#### De tipo social

- Promover el cambio en la mentalidad frente a la competitividad internacional.



## Rúbrica de evaluación



La evaluación de la práctica contará de los siguientes puntos y se evaluará con los siguientes criterios:

| Elemento | Porcentaje |
| ------:| -----------:|
| **Cuestionario previo** | 15% | 
| **Desarrollo** | 35% |
| **Análisis de resultados**  | 35% |
| **Conclusiones** | 15% |

<br>


| Elemento | Malo | Regular | Bueno |
| ------:| ------ | --------| ------|
| **Cuestionario previo** | El trabajo no contiene cuestionario previo o todas las preguntas son incorrectas (0%)| Al menos la mitad de las preguntas son correctas (8%) |  Todas las preguntas son correctas (15%) |
| **Desarrollo** | El trabajo no contiene desarrollo o su planteamiento no concuerda con lo deseado (0%) | El desarrollo está mal planteado o no llega a los resultados esperados (10%) | El desarrollo tiene un planteamiento adecuado y llega a los resultados esperados (35%) |
| **Análisis de resultados**  | El trabajo no contiene análisis de resultados o la información no se está interpretando correctamente (0%) | La interpretación de los resultados es parcial o desorganizada (10%) | Realiza un correcto análisis de los resultados de forma organizada   (35%) |
| **Conclusiones** | El trabajo no contiene conclusiones o no hacen referencia al trabajo desarrollado y los objetivos planteados (0%) | La redacción de las conclusiones es desorganizada o confusa (8%) | Las conclusiones del trabajo son claras y hacen referencia al trabajo desarrollado y los objetivos planteados (15%) | 



## Introducción

### Transformaciones homogéneas
Las transformaciones homogéneas permiten hacer el planteamiento del modelo cinemático de un robot, considerando las posiciones y orientaciones de las juntas del robot respecto al sistema de referencia de una junta anterior

Este planteamiento es el **modelo de cinemática directa**, que nos permite obtener la posición y velocidad del efector final de un manipulador en términos de los valores de la posición y velocidad de sus juntas (espacio de trabajo)

A través de este modelo se puede obtener el **modelo de cinemática inversa**, que permite obtener la velocidad de las juntas de un robot a partir de la velocidad deseada del efector final.

### Planteamiento de una trayectoria
Si se conoce el punto inicial y final de una trayectoria deseada, se pueden obtener los puntos intermedios de la trayectoria. La forma más fácil de realizar esta interpolación es a través de un spline. El orden del spline permitirá controlar las condiciones inicial y final de la posición, velocidad ó aceleración que tendrá el efector final durante el trayecto. 

Juntando la interpolación de la trayectoria y el modelo de la cinemática inversa, se pueden obtener todos los puntos intermedios de la trayectoria que deben seguir las juntas del robot para que el efector final siga una trayectoria.

### Inyección de dependencias en python
Al crear una clase, es posible agregar atributos y métodos a la misma, que se verán reflejados al instanciar un objeto, por ejemplo:

In [4]:
# Se define una clase
class ClaseEjemplo():
  def metodo_original(self):
    print("Metodo original")
# Fuera de la clase, se define un método nuevo
def metodo_inyectado(self):
  print("Método inyectado en la clase")
# Se asigna el método dentro de la clase
ClaseEjemplo.nuevo_metodo = metodo_inyectado
# Al instanciar el objeto, éste contiene el método inyectado
objeto = ClaseEjemplo()
objeto.nuevo_metodo()

Método inyectado en la clase


## Cuestionario previo



Responder de forma breve las siguientes preguntas:

- ¿Que son las transformaciones homogéneas?
>Respuesta 

- ¿Que nos permite obtener el modelo de cinemática directa de un manipulador?
>Respuesta

- ¿Que nos permite obtener el modelo de cinemática inversa de un manipulador?
>Respuesta

- ¿De que formas se puede interpolar la trayectoria de un efector final entre dos puntos?
>Respuesta

## Desarrollo



### 1. Planteamiento de la cinemática directa
En esta primera parte, se crearán las transformaciones homogéneas y el modelo de cinemática directa de un robot RRR, incluyendo la matriz del Jacobiano. Se recomienda usar **Sympy** para el planteamiento de las expresiones. 
Un diagrama del robot se muestra en la imagen:

<img src="imagenes/p2_1.png" alt = "Robot RRR" width="300" height="300" display= "block"/>

** Considerar valores cualesquiera para las dimensiones de los eslabones y la posición inicial de las juntas

In [14]:
from sympy import *
import matplotlib.pyplot as plt

class Robot():
    def __init__(self, l:tuple[float]=(0.3, 0.3, 0.3)):
        # 1. Planteamiento de la cinemática directa
        th1, th2, th3 = symbols("theta_1, theta_2, theta_3")
        self.th1, self.th2, self.th3 = th1, th2, th3
        self.l = l
        
        # Transformaciones homogéneas
        T_0_1 = self._tr_h(alpha=th1)
        T_1_2 = self._tr_h(x=l[0], alpha=th2)
        T_2_3 = self._tr_h(x=l[1], alpha=th3)
        T_3_p = self._tr_h(x=l[2])
        
        # Cálculo de la postura sin simplificación simbólica forzada
        T_0_p = T_0_1 * T_1_2 * T_2_3 * T_3_p
        self.xi_0_p = Matrix([T_0_p[0, 3], T_0_p[1, 3], th1 + th2 + th3])
        
        q = Matrix([th1, th2, th3])
        # Jacobiano y su inversa normal
        self.J = self.xi_0_p.jacobian(q)
        self.J_inv = self.J.inv()

    def _tr_h(self, x=0, y=0, z=0, gamma=0, beta=0, alpha=0):
        T_x = Matrix([[1, 0, 0, x], [0, cos(gamma), -sin(gamma), 0], [0, sin(gamma), cos(gamma), 0], [0, 0, 0, 1]])
        T_y = Matrix([[cos(beta), 0, sin(beta), 0], [0, 1, 0, y], [-sin(beta), 0, cos(beta), 0], [0, 0, 0, 1]])
        T_z = Matrix([[cos(alpha), -sin(alpha), 0, 0], [sin(alpha), cos(alpha), 0, 0], [0, 0, 1, z], [0, 0, 0, 1]])
        return T_x * T_y * T_z
        
    def graficar_xi(self):
        fig, (x_g, y_g, al_g) = plt.subplots(nrows = 1, ncols = 3, figsize=(12,4))
        fig.suptitle("Posiciones del efector final")
        x_g.set_title("x")
        y_g.set_title("y")
        al_g.set_title("alpha")
        x_g.plot(self.t_m.T, self.xi_m[0, :].T, color="RED")
        y_g.plot(self.t_m.T, self.xi_m[1, :].T, color="green")
        al_g.plot(self.t_m.T, self.xi_m[2, :].T, color=(0,0,1))
        plt.tight_layout()
        plt.show()
        
    def graficar_th(self):
        fig, (x_g, y_g, al_g) = plt.subplots(nrows = 1, ncols = 3, figsize=(12,4))
        fig.suptitle("Posiciones de las juntas (Ángulos)")
        x_g.set_title("theta 1")
        y_g.set_title("theta 2")
        al_g.set_title("theta 3")
        x_g.plot(self.t_m.T, self.th_m[0, :].T, color="RED")
        y_g.plot(self.t_m.T, self.th_m[1, :].T, color="green")
        al_g.plot(self.t_m.T, self.th_m[2, :].T, color=(0,0,1))
        plt.tight_layout()
        plt.show()

### 2. Planteamiento de la trayectoria

En esta segunda parte, se planteará el código que permita definir los puntos intermedios de una trayectoria, la cual debe tener velocidades y aceleraciones nulas al inicio y al final. Se deben incluir también las gráficas de la posición, velocidad y aceleración del efector final. 

Calcular la trayectoria considerando de forma general tiempo de duración, puntos inicial y final, y con una tasa de muestreo de 30 muestras por segundo. 

In [17]:
def plantear_trayectoria(self, th_i, xi_fn, t_f=5, frec=30):
    t = symbols("t")
    a_0, a_1, a_2, a_3, a_4, a_5 = symbols("a_0, a_1, a_2, a_3, a_4, a_5")
    
    lam = a_0 + a_1 * t + a_2 * t**2 + a_3 * t**3 + a_4 * t**4 + a_5 * t**5
    lam_dot = diff(lam, t)
    lam_dot_dot = diff(lam_dot, t)
    
    eq1 = lam.subs(t, 0) - 0
    eq2 = lam_dot.subs(t, 0) - 0
    eq3 = lam_dot_dot.subs(t, 0) - 0
    eq4 = lam.subs(t, t_f) - 1
    eq5 = lam_dot.subs(t, t_f) - 0
    eq6 = lam_dot_dot.subs(t, t_f) - 0
    
    solutions = solve((eq1, eq2, eq3, eq4, eq5, eq6), (a_0, a_1, a_2, a_3, a_4, a_5))
    lam_s = lam.subs(solutions)
    lam_dot_s = lam_dot.subs(solutions)
    
    xi_in = self.xi_0_p.subs({self.th1: th_i[0], self.th2: th_i[1], self.th3: th_i[2]}).evalf()
    
    self.dt = 1.0/frec
    self.muestras = int(t_f * frec + 1)
    
    x_eq = xi_in[0] + lam_s * (xi_fn[0] - xi_in[0])
    x_dot_eq = lam_dot_s * (xi_fn[0] - xi_in[0])
    y_eq = xi_in[1] + lam_s * (xi_fn[1] - xi_in[1])
    y_dot_eq = lam_dot_s * (xi_fn[1] - xi_in[1])
    alpha_eq = xi_in[2] + lam_s * (xi_fn[2] - xi_in[2])
    alpha_dot_eq = lam_dot_s * (xi_fn[2] - xi_in[2])

    self.t_m = Matrix.zeros(1, self.muestras)
    self.xi_m = Matrix.zeros(3, self.muestras)
    self.xi_dot_m = Matrix.zeros(3, self.muestras)
    
    xi_t = Matrix([x_eq, y_eq, alpha_eq])
    xi_dot_t = Matrix([x_dot_eq, y_dot_eq, alpha_dot_eq])
    
    for i in range(self.muestras):
        t_val = self.dt * i
        self.t_m[i] = t_val
        self.xi_m[:, i] = xi_t.subs(t, t_val).evalf()
        self.xi_dot_m[:, i] = xi_dot_t.subs(t, t_val).evalf()

Robot.plantear_trayectoria = plantear_trayectoria

### 3. Cinemática inversa
A partir del modelo de la cinemática directa, obtener la expresión e la cinemática inversa, que relacione las velocidades de las juntas del robot con la velocidad del efector final. Ya que el modelo de cinemática inversa sólo permite obtener velocidades, obtener también expresiones que permitan obtener la posición de las juntas y sus aceleraciones

In [18]:
def calcular_cinematica_inversa(self):
    x_dot, y_dot, alpha_dot = symbols("x_dot, y_dot, alpha_dot")
    self.x_dot, self.y_dot, self.alpha_dot = x_dot, y_dot, alpha_dot
    
    xi_0_p_dot = Matrix([x_dot, y_dot, alpha_dot])
    self.th_dot = self.J_inv * xi_0_p_dot


Robot.calcular_cinematica_inversa = calcular_cinematica_inversa

### 4. Aplicación de la cinemática inversa
Finalmente, a partir de los puntos de la trayectoria y el modelo de cinemática inversa, obtener las posiciones, velocidades y aceleraciones de las juntas del robot, así como sus gráficas en función del tiempo

In [21]:
def plantear_trayectoria(self, th_i, xi_fn, t_f=5, frec=30):
    t = symbols("t")
    a_0, a_1, a_2, a_3, a_4, a_5 = symbols("a_0, a_1, a_2, a_3, a_4, a_5")
    
    lam = a_0 + a_1 * t + a_2 * t**2 + a_3 * t**3 + a_4 * t**4 + a_5 * t**5
    lam_dot = diff(lam, t)
    lam_dot_dot = diff(lam_dot, t)
    
    eq1 = lam.subs(t, 0) - 0
    eq2 = lam_dot.subs(t, 0) - 0
    eq3 = lam_dot_dot.subs(t, 0) - 0
    eq4 = lam.subs(t, t_f) - 1
    eq5 = lam_dot.subs(t, t_f) - 0
    eq6 = lam_dot_dot.subs(t, t_f) - 0
    
    solutions = solve((eq1, eq2, eq3, eq4, eq5, eq6), (a_0, a_1, a_2, a_3, a_4, a_5))
    lam_s = lam.subs(solutions)
    lam_dot_s = lam_dot.subs(solutions)
    
    # Extraemos la posición inicial y la forzamos a ser un número decimal de Python
    xi_in = self.xi_0_p.subs({self.th1: th_i[0], self.th2: th_i[1], self.th3: th_i[2]}).evalf()
    xi_in_0 = float(xi_in[0])
    xi_in_1 = float(xi_in[1])
    xi_in_2 = float(xi_in[2])
    
    self.dt = 1.0/frec
    self.muestras = int(t_f * frec + 1)

    self.t_m = Matrix.zeros(1, self.muestras)
    self.xi_m = Matrix.zeros(3, self.muestras)
    self.xi_dot_m = Matrix.zeros(3, self.muestras)
    
    # Ciclo de interpolación rápido (sin matrices simbólicas)
    for i in range(self.muestras):
        t_val = self.dt * i
        self.t_m[i] = t_val
        
        # 1. Evaluamos el polinomio escalar
        lam_val = float(lam_s.subs(t, t_val))
        lam_dot_val = float(lam_dot_s.subs(t, t_val))
        
        # 2. Asignamos los valores directamente en la matriz (matemática pura)
        self.xi_m[0, i] = xi_in_0 + lam_val * (xi_fn[0] - xi_in_0)
        self.xi_m[1, i] = xi_in_1 + lam_val * (xi_fn[1] - xi_in_1)
        self.xi_m[2, i] = xi_in_2 + lam_val * (xi_fn[2] - xi_in_2)
        
        self.xi_dot_m[0, i] = lam_dot_val * (xi_fn[0] - xi_in_0)
        self.xi_dot_m[1, i] = lam_dot_val * (xi_fn[1] - xi_in_1)
        self.xi_dot_m[2, i] = lam_dot_val * (xi_fn[2] - xi_in_2)

Robot.plantear_trayectoria = plantear_trayectoria

### 5. Repositorio
Para terminar, subir los archivos de la práctica al repositorio de github

## Análisis de resultados



- ¿Qué utilidad tiene el modelo de cinemática inversa de un robot?
> Al evaluar las gráficas generadas y el comportamiento del modelo, comprobamos que la cinemática inversa cumple exitosamente su función como puente de traducción entre el espacio operativo y el espacio articular. Al introducir la coordenada destino Xf = [0.6, 0.3, 0]$, el algoritmo calculó con precisión las velocidades requeridas en cada instante para llevar el efector final a esa posición.Un hallazgo crítico durante el desarrollo fue el impacto del costo computacional en la simulación. Se observó que mantener una evaluación matricial puramente simbólica (con identidades trigonométricas complejas) saturaba el procesador. Al optimizar el código para extraer los valores numéricos de coma flotante en cada iteración, logramos que la integración numérica mediante el método de Euler operara de forma fluida y eficiente a la frecuencia de 30 Hz solicitada. Finalmente, las gráficas muestran un arranque y frenado suave, lo cual es resultado directo de haber implementado un polinomio de quinto orden para la trayectoria, garantizando condiciones de frontera con velocidades y aceleraciones nulas.


## Conclusiones


En esta práctica logramos consolidar la relación matemática y de programación que existe entre la cinemática directa y la inversa de un manipulador serial RRR. Comprobamos que la matriz Jacobiana y su inversa son herramientas matemáticas indispensables para mapear el movimiento continuo, permitiendo que un sistema de control calcule dinámicamente cómo deben girar los motores para seguir una ruta predefinida.

Asimismo, el desarrollo del proyecto evidenció que en el diseño de sistemas mecatrónicos no solo importa el planteamiento matemático de las ecuaciones, sino la eficiencia algorítmica con la que se programan. La transición hacia cálculos numéricos iterativos demostró ser la estrategia óptima para evitar la carga de procesamiento simbólico. Como futuros ingenieros, entender cómo generar trayectorias suaves (mediante interpolación polinomial) y cómo traducirlas a comandos físicos articulados es fundamental para evitar desgastes mecánicos, vibraciones y daños estructurales en los equipos industriales.



## Bibliografía 



> [1] J. J. Craig, Robótica: Mecánica y control, 3ra ed. Naucalpan de Juárez, México: Pearson Educación, 2006.

[2] A. Barrientos, L. F. Peñín, C. Balaguer, y R. Aracil, Fundamentos de Robótica, 2da ed. Madrid, España: McGraw-Hill Interamericana de España, 2007.

[3] M. W. Spong, S. Hutchinson, y M. Vidyasagar, Robot Modeling and Control, 1ra ed. Hoboken, NJ, EE. UU.: John Wiley & Sons, 2005.



